# Ramy Sentiment Fine-Tuning on H100 (Leakage-Free)

This notebook fine-tunes an Arabic transformer model using:
- `data/augmented/Ramy_data_train_target_1500.csv`
- `data/augmented/Ramy_data_val_target_1500.csv`

### Leakage fixes applied
| # | Original issue | Fix |
|---|---|---|
| 1 | `load_best_model_at_end=True` used `val_ds` to pick the best checkpoint | Epoch checkpointing **disabled** — train for fixed epochs, no val-driven selection |
| 2 | TTA ran on `val_ds` (same set used during training decisions) | TTA **moved to a separate held-out test split** carved from val_df before training |
| 3 | Per-class threshold grid-search run on `val_ds`, score reported on same set | Thresholds tuned on `val_ds` (now a true dev set), **reported score on held-out test split** |

**Split strategy:** 80% of the original val CSV → `dev_df` (used by Trainer eval callback + threshold tuning), 20% → `test_df` (untouched until final evaluation).

## Low-Data Boosting Strategy

This version combines:
1. Few-shot style text augmentation to increase lexical diversity.
2. Teacher-to-student self-training with high-confidence pseudo-labels.
3. TTA inference averaging — **on the held-out test split only**.

These steps improve macro-F1 in low-data settings without leaking evaluation signal into training decisions.

## 1) Install dependencies (cluster-safe)

In [1]:
import sys
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q pandas numpy scikit-learn transformers datasets accelerate safetensors

## 2) Imports and config

In [2]:
from pathlib import Path
import csv
import json
import random
import re
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "aubmindlab/bert-base-arabertv02"
TRAIN_PATH = Path("./Ramy_data_train_target_1500.csv")
VAL_PATH   = Path("./Ramy_data_val_target_1500.csv")
UNLABELED_POOL_PATH = Path("data/Ramy_data.csv")
OUTPUT_DIR = Path("models/checkpoints/h100_arabert_ft")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Low-data boosters
USE_FEW_SHOT_AUG           = True
AUG_MULTIPLIER             = 1
USE_SELF_TRAINING          = True
PSEUDO_LABEL_MIN_CONFIDENCE = 0.92
MAX_PSEUDO_SAMPLES         = 12000
USE_TTA_EVAL               = True   # TTA applied only on held-out test split

# Leakage-free split: carve a held-out TEST set from the original val CSV
# 80% → dev (used by Trainer callback + threshold tuning)
# 20% → test (never seen until final evaluation)
TEST_SPLIT_RATIO = 0.20

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.12.6 (main, Sep 27 2024, 06:10:12) [GCC 12.2.0]
Torch: 2.8.0+cu129
CUDA available: True
GPU: NVIDIA H100 80GB HBM3


## 3) Load and split data

In [3]:
def load_semicolon_csv(path: Path) -> pd.DataFrame:
    rows = []
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f, delimiter=';')
        for parts in reader:
            if len(parts) < 3:
                continue
            text    = parts[0].strip()
            product = parts[1].strip()
            label   = parts[2].strip().lower()
            if not text or not label:
                continue
            rows.append({"text": text, "product": product, "sentiment": label})
    return pd.DataFrame(rows)

def load_unlabeled_pool(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["text", "product"])
    rows = []
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f, delimiter=';')
        for parts in reader:
            if len(parts) < 1:
                continue
            text    = parts[0].strip()
            product = parts[1].strip() if len(parts) > 1 else ""
            if text:
                rows.append({"text": text, "product": product})
    return pd.DataFrame(rows)

def few_shot_augment_text(text: str) -> list[str]:
    t = str(text)
    variants = [t]
    v1 = re.sub(r"\s+", " ", t).strip()
    v2 = re.sub(r"(.)\1{2,}", r"\1\1", t)
    v3 = t.replace("؟", "?").replace("،", ",")
    swaps = {"بزاف": "كثير", "مليح": "جيد", "خايب": "سيء",
             "bon": "جيد", "mauvais": "سيء", "prix": "السعر"}
    v4 = t
    for src, dst in swaps.items():
        v4 = v4.replace(src, dst)
    for v in [v1, v2, v3, v4]:
        v = v.strip()
        if v and v not in variants:
            variants.append(v)
    return variants[: 1 + AUG_MULTIPLIER]

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'Train file not found: {TRAIN_PATH}')
if not VAL_PATH.exists():
    raise FileNotFoundError(f'Validation file not found: {VAL_PATH}')

train_df        = load_semicolon_csv(TRAIN_PATH)
full_val_df     = load_semicolon_csv(VAL_PATH)
unlabeled_pool_df = load_unlabeled_pool(UNLABELED_POOL_PATH)

if train_df.empty or full_val_df.empty:
    raise ValueError('One of the datasets is empty after parsing.')

# ── LEAKAGE FIX: split val CSV into dev (for training decisions) and test (held-out) ──
dev_df, test_df = train_test_split(
    full_val_df,
    test_size=TEST_SPLIT_RATIO,
    stratify=full_val_df['sentiment'],
    random_state=SEED,
)
dev_df  = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Augment only the training set
if USE_FEW_SHOT_AUG:
    aug_rows = []
    for row in train_df.itertuples(index=False):
        for v in few_shot_augment_text(row.text):
            aug_rows.append({"text": v, "product": row.product, "sentiment": row.sentiment})
    train_df = pd.DataFrame(aug_rows).drop_duplicates(subset=["text", "sentiment"]).reset_index(drop=True)

print(f'Train rows  : {len(train_df)}')
print(f'Dev rows    : {len(dev_df)}   ← Trainer callback + threshold tuning')
print(f'Test rows   : {len(test_df)}  ← Final evaluation only (untouched until end)')
print('Train label distribution:\n', train_df['sentiment'].value_counts())
print('Dev label distribution:\n',   dev_df['sentiment'].value_counts())
print('Test label distribution:\n',  test_df['sentiment'].value_counts())

Train rows  : 9481
Dev rows    : 324   ← Trainer callback + threshold tuning
Test rows   : 81  ← Final evaluation only (untouched until end)
Train label distribution:
 sentiment
question       2430
negative       1920
positive       1760
improvement    1694
neutral        1677
Name: count, dtype: int64
Dev label distribution:
 sentiment
positive       100
negative        62
neutral         56
question        54
improvement     52
Name: count, dtype: int64
Test label distribution:
 sentiment
positive       25
negative       16
neutral        14
question       13
improvement    13
Name: count, dtype: int64


## 4) Build label mappings

In [4]:
labels    = sorted(train_df['sentiment'].unique().tolist())
label2id  = {lbl: i for i, lbl in enumerate(labels)}
id2label  = {i: lbl for lbl, i in label2id.items()}

for df in [train_df, dev_df, test_df]:
    df.drop(df[~df['sentiment'].isin(labels)].index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df['label_id'] = df['sentiment'].map(label2id)

print('Labels:', labels)
print('Num labels:', len(labels))

Labels: ['improvement', 'negative', 'neutral', 'positive', 'question']
Num labels: 5


## 5) Tokenization

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[['text', 'label_id']], preserve_index=False)
    ds = ds.map(lambda batch: tokenizer(batch['text'], truncation=True, max_length=256), batched=True)
    ds = ds.rename_column('label_id', 'labels')
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    return ds

def to_hf_dataset_unlabeled(df: pd.DataFrame) -> Dataset:
    """For pools without a label column."""
    ds = Dataset.from_pandas(df[['text']], preserve_index=False)
    ds = ds.map(lambda batch: tokenizer(batch['text'], truncation=True, max_length=256), batched=True)
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])
    return ds

train_ds = to_hf_dataset(train_df)
dev_ds   = to_hf_dataset(dev_df)    # used by Trainer eval callback only
# test_ds intentionally NOT created here — built just-in-time during final evaluation

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Prepared tokenized datasets.')

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/9481 [00:00<?, ? examples/s]

Map:   0%|          | 0/324 [00:00<?, ? examples/s]

Prepared tokenized datasets.


## 6) Model and Trainer

In [6]:
def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
    }

def make_training_args(output_subdir: str):
    return TrainingArguments(
        output_dir=str(OUTPUT_DIR / output_subdir),
        num_train_epochs=4,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_steps=50,
        # ── LEAKAGE FIX 1 ──────────────────────────────────────────────────────
        # eval_strategy='no' means the Trainer never calls predict() on dev_ds
        # mid-training, so dev_ds cannot influence which checkpoint is kept.
        # We train for the full fixed number of epochs.
        # If you want early stopping in the future, use a SEPARATE early-stop
        # split that is NOT the same split you report final numbers on.
        eval_strategy='no',
        save_strategy='no',
        load_best_model_at_end=False,   # must be False when save_strategy='no'
        # ───────────────────────────────────────────────────────────────────────
        fp16=torch.cuda.is_available(),
        report_to='none',
        seed=SEED,
    )

def make_trainer(model, train_dataset, output_subdir: str):
    return Trainer(
        model=model,
        args=make_training_args(output_subdir),
        train_dataset=train_dataset,
        # No eval_dataset passed — dev_ds is only used in the explicit
        # post-training evaluation cell below.
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

teacher_model   = build_model()
teacher_trainer = make_trainer(teacher_model, train_ds, 'teacher_output')
print('Teacher trainer ready.')

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_115/4237656526.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(
Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Teacher trainer ready.


## 7) Train

In [7]:
# 1) Train teacher (fixed epochs, no val-driven checkpoint selection)
teacher_result = teacher_trainer.train()
print('Teacher training done.')

final_trainer = teacher_trainer
student_used  = False

# 2) Self-training from high-confidence pseudo-labels
if USE_SELF_TRAINING and not unlabeled_pool_df.empty:
    pool_df = unlabeled_pool_df.copy()
    pool_df['text'] = pool_df['text'].astype(str)
    pool_df = pool_df[pool_df['text'].str.len() > 0].drop_duplicates(subset=['text']).reset_index(drop=True)

    if len(pool_df) > MAX_PSEUDO_SAMPLES:
        pool_df = pool_df.sample(MAX_PSEUDO_SAMPLES, random_state=SEED).reset_index(drop=True)

    pool_ds  = to_hf_dataset_unlabeled(pool_df)
    pred     = teacher_trainer.predict(pool_ds)
    probs    = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    conf     = probs.max(axis=1)
    pred_ids = probs.argmax(axis=1)

    keep      = conf >= PSEUDO_LABEL_MIN_CONFIDENCE
    pseudo_df = pool_df.loc[keep, ['text']].copy()

    if not pseudo_df.empty:
        pseudo_df['sentiment'] = [id2label[int(i)] for i in pred_ids[keep]]
        pseudo_df['product']   = 'pseudo'
        pseudo_df['label_id']  = pseudo_df['sentiment'].map(label2id)

        student_train_df = pd.concat(
            [train_df, pseudo_df[['text', 'product', 'sentiment', 'label_id']]],
            ignore_index=True,
        ).drop_duplicates(subset=['text', 'sentiment']).reset_index(drop=True)

        student_ds      = to_hf_dataset(student_train_df)
        student_model   = build_model()
        student_trainer = make_trainer(student_model, student_ds, 'student_output')
        student_trainer.train()
        print('Student training done. Pseudo samples used:', len(pseudo_df))

        final_trainer = student_trainer
        student_used  = True
    else:
        print('No pseudo-labeled samples passed confidence threshold.')
else:
    print('Self-training skipped.')

print('Using student model:' if student_used else 'Using teacher model:', student_used)

Step,Training Loss
50,1.574300
100,1.137900
150,0.614100
200,0.328800
250,0.195000
300,0.156100
350,0.081500
400,0.094000
450,0.048300
500,0.048800


Teacher training done.
Self-training skipped.
Using teacher model: False


## 8) Threshold tuning on dev set (leakage-free)

In [8]:
# ── LEAKAGE FIX 2 & 3 combined ──────────────────────────────────────────────
# TTA and threshold tuning are performed on dev_df / dev_ds.
# This is acceptable because dev_ds was NOT used during training
# (eval_strategy='no' above).  dev_ds plays the role of a true
# held-out development set: train never saw it, but we are allowed
# to tune post-processing decisions (TTA, thresholds) on it.
# Final numbers are reported on test_df (next cell) which is never
# touched here.
# ─────────────────────────────────────────────────────────────────────────────

def get_logits_with_tta(trainer, df: pd.DataFrame) -> np.ndarray:
    """Run prediction (optionally with TTA) and return averaged logits."""
    if USE_TTA_EVAL:
        tta_logits = []
        for suffix in ["", " !", " ؟"]:
            df_tta = df.copy()
            df_tta['text'] = df_tta['text'].astype(str) + suffix
            ds_tta = to_hf_dataset(df_tta)
            p = trainer.predict(ds_tta)
            tta_logits.append(p.predictions)
        return np.mean(np.stack(tta_logits, axis=0), axis=0)
    else:
        ds  = to_hf_dataset(df)
        p   = trainer.predict(ds)
        return p.predictions

def apply_thresholds(prob_matrix: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    adjusted = prob_matrix / np.clip(thresholds.reshape(1, -1), 1e-8, None)
    return np.argmax(adjusted, axis=1)

def tune_thresholds(logits: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    """Grid-search per-class thresholds that maximise macro-F1 on the given split."""
    exp_l  = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs  = exp_l / np.clip(exp_l.sum(axis=1, keepdims=True), 1e-8, None)

    best_thresholds = np.ones(len(labels), dtype=float)
    best_pred = np.argmax(probs, axis=1)
    best_f1   = f1_score(y_true, best_pred, average='macro')

    grid = np.arange(0.60, 1.46, 0.05)
    for class_idx in range(len(labels)):
        for t in grid:
            trial       = best_thresholds.copy()
            trial[class_idx] = t
            pred_trial  = apply_thresholds(probs, trial)
            f1_trial    = f1_score(y_true, pred_trial, average='macro')
            if f1_trial > best_f1:
                best_f1                      = f1_trial
                best_thresholds[class_idx]   = t
        best_pred = apply_thresholds(probs, best_thresholds)
        best_f1   = f1_score(y_true, best_pred, average='macro')

    return best_thresholds, probs

# --- Run on dev set to find thresholds ---
dev_logits = get_logits_with_tta(final_trainer, dev_df)
dev_y_true = dev_df['label_id'].values

best_thresholds, dev_probs = tune_thresholds(dev_logits, dev_y_true)

dev_pred_base = np.argmax(dev_probs, axis=1)
dev_pred_tuned = apply_thresholds(dev_probs, best_thresholds)

print('Dev set (used only for tuning — NOT the reported score):')
print(f'  Base macro-F1   : {f1_score(dev_y_true, dev_pred_base,  average="macro"):.4f}')
print(f'  Tuned macro-F1  : {f1_score(dev_y_true, dev_pred_tuned, average="macro"):.4f}')
print('Tuned thresholds:',
      {labels[i]: round(float(best_thresholds[i]), 2) for i in range(len(labels))})

Map:   0%|          | 0/324 [00:00<?, ? examples/s]

Map:   0%|          | 0/324 [00:00<?, ? examples/s]

Map:   0%|          | 0/324 [00:00<?, ? examples/s]

Dev set (used only for tuning — NOT the reported score):
  Base macro-F1   : 0.9176
  Tuned macro-F1  : 0.9176
Tuned thresholds: {'improvement': 1.0, 'negative': 1.0, 'neutral': 1.0, 'positive': 1.0, 'question': 1.0}


## 9) Final evaluation on held-out TEST set

In [9]:
# ── This is the only place test_df is ever used ──────────────────────────────
# Thresholds were tuned on dev_df above. We now apply them to test_df and
# report those numbers as the honest estimate of generalisation performance.
# ─────────────────────────────────────────────────────────────────────────────

test_logits = get_logits_with_tta(final_trainer, test_df)
test_y_true = test_df['label_id'].values

exp_l_test  = np.exp(test_logits - test_logits.max(axis=1, keepdims=True))
test_probs  = exp_l_test / np.clip(exp_l_test.sum(axis=1, keepdims=True), 1e-8, None)

test_pred_base  = np.argmax(test_probs, axis=1)
test_pred_tuned = apply_thresholds(test_probs, best_thresholds)

metrics = {
    'split': 'held_out_test',
    'n_samples': len(test_df),
    'accuracy': float(accuracy_score(test_y_true, test_pred_tuned)),
    'f1_macro': float(f1_score(test_y_true, test_pred_tuned, average='macro')),
    'f1_macro_base_no_threshold': float(f1_score(test_y_true, test_pred_base, average='macro')),
    'labels': labels,
    'classification_report': classification_report(
        test_y_true, test_pred_tuned,
        labels=list(range(len(labels))), target_names=labels, zero_division=0,
    ),
    'confusion_matrix': confusion_matrix(
        test_y_true, test_pred_tuned,
        labels=list(range(len(labels))),
    ).tolist(),
    'used_self_training': bool(student_used),
    'tta_used': USE_TTA_EVAL,
    'threshold_optimization': {
        'enabled': True,
        'tuned_on': 'dev_split',
        'thresholds': {labels[i]: float(best_thresholds[i]) for i in range(len(labels))},
    },
}

print('=' * 55)
print('FINAL RESULTS (held-out test set — honest estimate)')
print('=' * 55)
print(f'  Accuracy  : {metrics["accuracy"]:.4f}')
print(f'  Macro-F1  : {metrics["f1_macro"]:.4f}  (with tuned thresholds)')
print(f'  Macro-F1  : {metrics["f1_macro_base_no_threshold"]:.4f}  (base argmax, no thresholds)')
print()
print(metrics['classification_report'])

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

FINAL RESULTS (held-out test set — honest estimate)
  Accuracy  : 0.9506
  Macro-F1  : 0.9437  (with tuned thresholds)
  Macro-F1  : 0.9437  (base argmax, no thresholds)

              precision    recall  f1-score   support

 improvement       1.00      0.85      0.92        13
    negative       0.94      1.00      0.97        16
     neutral       0.92      0.86      0.89        14
    positive       0.96      1.00      0.98        25
    question       0.93      1.00      0.96        13

    accuracy                           0.95        81
   macro avg       0.95      0.94      0.94        81
weighted avg       0.95      0.95      0.95        81



## 10) Save model, tokenizer, and metrics

In [10]:
final_model_dir = OUTPUT_DIR / 'final_model'
final_model_dir.mkdir(parents=True, exist_ok=True)

final_trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

metrics_path = OUTPUT_DIR / 'test_metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

label_map_path = OUTPUT_DIR / 'label_mapping.json'
with label_map_path.open('w', encoding='utf-8') as f:
    json.dump({'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, f, ensure_ascii=False, indent=2)

strategy_path = OUTPUT_DIR / 'training_strategy.json'
with strategy_path.open('w', encoding='utf-8') as f:
    json.dump({
        'few_shot_augmentation': USE_FEW_SHOT_AUG,
        'self_training': USE_SELF_TRAINING,
        'pseudo_label_min_confidence': PSEUDO_LABEL_MIN_CONFIDENCE,
        'tta_eval': USE_TTA_EVAL,
        'used_self_training': bool(student_used),
        'leakage_fixes': [
            'eval_strategy=no: val set never drives checkpoint selection',
            'TTA applied only on held-out test split',
            'threshold tuning on dev split; scores reported on separate test split',
        ],
        'threshold_optimization': metrics['threshold_optimization'],
    }, f, ensure_ascii=False, indent=2)

thresholds_path = OUTPUT_DIR / 'class_thresholds.json'
with thresholds_path.open('w', encoding='utf-8') as f:
    json.dump(metrics['threshold_optimization'], f, ensure_ascii=False, indent=2)

print('Saved model folder:', final_model_dir)
print('Saved test metrics:', metrics_path)
print('Saved label mapping:', label_map_path)
print('Saved strategy file:', strategy_path)
print('Saved thresholds file:', thresholds_path)

Saved model folder: models/checkpoints/h100_arabert_ft/final_model
Saved test metrics: models/checkpoints/h100_arabert_ft/test_metrics.json
Saved label mapping: models/checkpoints/h100_arabert_ft/label_mapping.json
Saved strategy file: models/checkpoints/h100_arabert_ft/training_strategy.json
Saved thresholds file: models/checkpoints/h100_arabert_ft/class_thresholds.json


## 11) Export a single zip for easy download

In [11]:
import shutil

zip_base = OUTPUT_DIR / 'ramy_h100_finetuned_model'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(final_model_dir))
print('Download this zip:', zip_path)

Download this zip: /root/models/checkpoints/h100_arabert_ft/ramy_h100_finetuned_model.zip


## 12) Quick reload test

In [12]:
from transformers import pipeline

clf = pipeline(
    task='text-classification',
    model=str(final_model_dir),
    tokenizer=str(final_model_dir),
    device=0 if torch.cuda.is_available() else -1,
)

sample_text = 'عصير رامي بنين والسعر مناسب جدا'
print(clf(sample_text))

Device set to use cuda:0


[{'label': 'positive', 'score': 0.9932097792625427}]
